In [0]:
spark.conf.set(
    "fs.azure.account.key.stdeltalakedemo.dfs.core.windows.net",
    "00000000000000000000000000000000000000000000000000000000000000000000000000000000000000==")

In [0]:
# Deduplicate using window
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col, lower, trim
from delta.tables import DeltaTable
from pyspark.sql.utils import AnalysisException

def is_valid_delta_table(path: str) -> bool:
    try:
        # Ensure directory exists
        entries = dbutils.fs.ls(path)

        # Explicitly check for _delta_log directory
        has_delta_log = any(f.name.rstrip("/") == "_delta_log" for f in entries)

        if not has_delta_log:
            return False

        # Validate Delta metadata
        return DeltaTable.isDeltaTable(spark, path)

    except AnalysisException:
        return False
    except FileNotFoundError:
        return False

# Define widgets (ADF will inject values here)
dbutils.widgets.text("p_trainee_id", "")
dbutils.widgets.text("p_bronze_path", "")
dbutils.widgets.text("p_silver_path", "")

# Retrieve parameter values
p_trainee_id = dbutils.widgets.get("p_trainee_id")
bronze_path  = dbutils.widgets.get("p_bronze_path")
silver_path  = dbutils.widgets.get("p_silver_path")

# Read bronze layer
bronze_df = spark.read.format("delta").load(bronze_path)

# Clean data
clean_df = bronze_df.filter((col("amount") > 0) & ~(lower(trim(col("status"))).like("%invalid%")))

windowSpec = Window.partitionBy("order_id").orderBy(col("order_time").desc())

dedup_df = (
    clean_df
      .withColumn("rn", row_number().over(windowSpec))
      .filter("rn = 1")
      .drop("rn")
)

# Cast and select columns
silver_df = dedup_df.select(
col("order_id").cast("int"),
col("customer_id"),
col("amount").cast("double"),
col("status"),
col("order_time")
)

# Write to silver
if is_valid_delta_table(silver_path):
    #check table exits
    silver_table = DeltaTable.forPath(spark, silver_path)
    
    (silver_table.alias("target")
     .merge(
        silver_df.alias("source"),
        "target.order_id = source.order_id" # Match on order_id
     )
     .whenMatchedUpdateAll() 
     .whenNotMatchedInsertAll() 
     .execute())
else:
    silver_df.write.format("delta").mode("overwrite").save(silver_path)